In [ ]:
!pip install -q jedi

!pip install "setuptools<82" -q

!pip install -q transformers datasets evaluate accelerate sentencepiece protobuf

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer

raw_dataset = load_dataset("json", data_files="/content/dataset_tokens_labeled.json")

label_list = ["O", "B-PER", "I-PER"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

dataset = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)

model_checkpoint = "efontes/efontes-ner"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )

    labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        tag_ids = [label2id[t] for t in tags]
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag_ids[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("Dataset ready:")
print(tokenized_datasets)

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        p_val for (p_seq, l_seq) in zip(predictions, labels)
        for (p_val, l_val) in zip(p_seq, l_seq) if l_val != -100
    ]
    true_labels = [
        l_val for (p_seq, l_seq) in zip(predictions, labels)
        for (p_val, l_val) in zip(p_seq, l_seq) if l_val != -100
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="weighted", zero_division=0
    )
    acc = accuracy_score(true_labels, true_predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)

MODEL_CHECKPOINT = "efontes/efontes-ner"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

label2id = {"O": 0, "B-PER": 1, "I-PER": 2}
id2label = {0: "O", 1: "B-PER", 2: "I-PER"}

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )

    labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        tag_ids = [label2id[t] if isinstance(t, str) else t for t in tags]
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag_ids[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./efontes-ner-person-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    push_to_hub=False,
    hub_model_id="antondep2/efontes-ner-person"
)

split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

test_text = "coram Officio stans personaliter Andreas Simonowicz Civis Leopolien recognovit"
print(ner(test_text))

In [ ]:
from transformers import pipeline

model_id = "antondep2/efontes-ner-person"

ner = pipeline(
    "token-classification",
    model=model_id,
    tokenizer=model_id,
    aggregation_strategy="none"
)

test_latin_text = (
    "Anno Domini millesimo quadringentesimo septuagesimo secundo, feria quarta proxima post festum Sancti Adalberti martyris. Coram honorabilibus et famatis dominis, videlicet Nicolao dicto Schepeler proconsule, Petro carnifice, Johanne aurifabro et Stanislao notario, consulibus ac scabinis civitatis Leopoliensis in praetorio pro tribunali sedentibus, personaliter comparuit providus vir Michael textor, civis ac conterraneus noster ibidem in Leopoli residens."

"Ex adverso vero constitutus est discretus Bartholomeus cerdo, qui praefatum Michaelem coram officio consulari ad instantiam iuris vocari fecerat. Idem Bartholomeus querulando proposuit contra eundem Michaelem textorem, asserens quomodo ipse Michael anno praeterito circa festum Nativitatis Sancti Johannis Baptistae decem ulnas panni coloniensis coloris glauci, necnon duas pelles ceruinas pro pretio quatuor marcarum monetae et numeri Regni Poloniae communis et currentis sibi tradiderit et vendiderit. Quam quidem summam pecuniarum praefatus Michael infra tempus trium mensium tunc proxime sequentium persolvere promiserat et spoponderat bona fide, fideiussione quoque interposita per Jacobum pannicidam. Nihilominus praefatus Michael, debiti sui minime memor, solutionem huiusmodi pecuniarum contra iustitiam facere distulit et in praesens usque recusat, in grave damnum et iacturam ipsius actoris."

"Reus vero Michael textor, audita propositione et petitione adversarii sui, fassus est coram iudicio se quidem dictum pannum et pelles recepisse, attamen asseruit duas marcas de eadem summa per manus uxoris suae Annae praedicto Bartholomeo iam antea fuisse integre persolutas, super quo probationem legitimam per testes fidedignos inducere se obligavit."

"Super quibus domini consules et scabini, auditis partium rationibus et confessionibus, decreverunt et terminarunt: Quod praefatus Michael textor tenetur infra quatuordecim dies a die praesentium numerandos duas marcas restantis debiti una cum expensis iudicii praefato Bartholomeo satisfacere, de residuis vero duabus marcis purgationem legalem facere debet. Actum in Leopoli, praesentibus testibus fidedignis Gregorio aurifabro et Andrea zonario."
)

raw_predictions = ner(test_latin_text)

entities = []
current = None

for token in raw_predictions:
    if "PER" in token["entity"]:
        if current is None:
            current = {
                "start": token["start"],
                "end": token["end"],
                "scores": [token["score"]]
            }
        else:
            if token["start"] <= current["end"] + 1:
                current["end"] = token["end"]
                current["scores"].append(token["score"])
            else:
                entities.append(current)
                current = {
                    "start": token["start"],
                    "end": token["end"],
                    "scores": [token["score"]]
                }
    else:
        if current:
            entities.append(current)
            current = None

if current:
    entities.append(current)

print(f"Total Words Tested: {len(test_latin_text.split())}\n")
print(f"{'Extracted Person Entity':<30} | {'Confidence':<10}")
print("-" * 45)
for ent in entities:
    name_str = test_latin_text[ent["start"]:ent["end"]].strip()
    avg_score = sum(ent["scores"]) / len(ent["scores"])
    print(f"{name_str:<30} | {avg_score:.2%}")

In [ ]:
import os
import re
import time
from pathlib import Path
import torch
from transformers import AutoConfig, pipeline

DATA_DIR = Path("/content/test_extracted") if Path("/content/test_extracted").exists() else Path("test_extracted")
BASE_MODEL_ID = "efontes/efontes-ner"
device = 0 if torch.cuda.is_available() else -1

TAG_PAIR_RE = re.compile(r"<per>(.*?)</?per>", re.DOTALL | re.IGNORECASE)

test_files = sorted(DATA_DIR.glob("*.txt"))
if not test_files:
    raise FileNotFoundError(f"No .txt files found in {DATA_DIR}.")

corpus = []
for f in test_files:
    raw = f.read_text(encoding="utf-8", errors="replace")
    if not raw.strip():
        continue

    gold_names = []
    for match in TAG_PAIR_RE.finditer(raw):
        clean_name = " ".join(match.group(1).split()).strip().lower()
        if clean_name:
            gold_names.append(clean_name)

    clean_text = re.sub(r"</?per>", " ", raw, flags=re.IGNORECASE)
    clean_text = " ".join(clean_text.split())

    corpus.append({
        "filename": f.name,
        "text": clean_text,
        "gold": gold_names
    })

total_words = sum(len(doc["text"].split()) for doc in corpus)
total_gold = sum(len(doc["gold"]) for doc in corpus)
print(f"Loaded {len(corpus)} test documents ({total_words} words, {total_gold} annotated person names).\n")


def get_target_labels(config):
    id2label = config.id2label
    targets = set()
    for idx, name in id2label.items():
        uname = str(name).upper()
        if any(k in uname for k in ["PER", "PERS", "NAME"]):
            targets.add(name)
    if not targets:
        targets = {name for idx, name in id2label.items() if idx != 0 and name != "O"}
    return targets

def merge_entity_fragments(raw_predictions, text, valid_labels):
    entities = []
    current = None

    for tok in raw_predictions:
        entity_tag = tok.get("entity", tok.get("entity_group", ""))
        if entity_tag in valid_labels or any(k in entity_tag.upper() for k in ["PER", "PERS"]):
            start, end = tok["start"], tok["end"]
            score = tok["score"]

            if current is None:
                current = {"start": start, "end": end, "scores": [score]}
            else:
                gap = text[current["end"]:start]
                if start <= current["end"] + 1 or gap.strip() == "":
                    current["end"] = end
                    current["scores"].append(score)
                else:
                    entities.append(current)
                    current = {"start": start, "end": end, "scores": [score]}
        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    merged_names = []
    for ent in entities:
        span_str = text[ent["start"]:ent["end"]].strip()
        if len(span_str) > 1:
            merged_names.append(span_str)

    return merged_names


def benchmark_model(model_or_id, tokenizer_obj, display_name: str):
    print(f"--> Initializing pipeline for: {display_name}...")
    pipe = pipeline(
        "token-classification",
        model=model_or_id,
        tokenizer=tokenizer_obj,
        device=device,
        aggregation_strategy="none"
    )

    valid_labels = get_target_labels(pipe.model.config)
    print(f"    Recognized person labels: {valid_labels}")

    t0 = time.perf_counter()
    all_preds = []
    per_doc_preds = {}
    true_positives = 0

    for doc in corpus:
        text = doc["text"]
        words = text.split()
        chunk_size = 250
        doc_preds = []

        for i in range(0, len(words), chunk_size):
            chunk = " ".join(words[i:i + chunk_size])
            raw_tokens = pipe(chunk)
            chunk_merged = merge_entity_fragments(raw_tokens, chunk, valid_labels)
            doc_preds.extend(chunk_merged)

        all_preds.extend(doc_preds)
        per_doc_preds[doc["filename"]] = doc_preds

        matched_indices = set()
        for gold in doc["gold"]:
            gold_compact = re.sub(r"[^\w]", "", gold.lower())
            for idx, pred in enumerate(doc_preds):
                if idx in matched_indices:
                    continue
                pred_compact = re.sub(r"[^\w]", "", pred.lower())
                if gold_compact in pred_compact or pred_compact in gold_compact:
                    true_positives += 1
                    matched_indices.add(idx)
                    break

    elapsed = time.perf_counter() - t0
    pred_count = len(all_preds)
    precision = true_positives / pred_count if pred_count > 0 else 0.0
    recall = true_positives / total_gold if total_gold > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    wps = total_words / elapsed if elapsed > 0 else 0.0

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "detected": pred_count,
        "true_positives": true_positives,
        "time": elapsed,
        "speed": wps,
        "doc_preds": per_doc_preds
    }

res_base = benchmark_model(BASE_MODEL_ID, BASE_MODEL_ID, "Base (efontes-ner)")
res_retrained = benchmark_model(trainer.model, tokenizer, "Retrained")

print("\n" + "=" * 76)
print(f"{'Metric':<26} | {'Base (efontes-ner)':<22} | {'Retrained':<22}")
print("-" * 76)
print(f"{'F1 Score':<26} | {res_base['f1']:.4f}{'':<16} | {res_retrained['f1']:.4f}")
print(f"{'Precision':<26} | {res_base['precision']:.4f}{'':<16} | {res_retrained['precision']:.4f}")
print(f"{'Recall':<26} | {res_base['recall']:.4f}{'':<16} | {res_retrained['recall']:.4f}")
print(f"{'True Positives':<26} | {res_base['true_positives']:<22} | {res_retrained['true_positives']:<22}")
print(f"{'Total Entities Found':<26} | {res_base['detected']:<22} | {res_retrained['detected']:<22}")
print(f"{'Total Time (s)':<26} | {res_base['time']:.2f} s{'':<15} | {res_retrained['time']:.2f} s")
print(f"{'Throughput (words/sec)':<26} | {res_base['speed']:.1f} w/s{'':<13} | {res_retrained['speed']:.1f} w/s")
print("=" * 76)

print("\n" + "=" * 76)
print("EXTRACTED ENTITIES PREVIEW (Retrained Model vs Ground Truth)")
print("=" * 76)
for doc in corpus[:2]:
    print(f"\n[File: {doc['filename']}]")
    print(f"  • Ground Truth Target ({len(doc['gold'])}):")
    print("    " + ", ".join(f"'{g}'" for g in doc['gold'][:6]) + ("..." if len(doc['gold']) > 6 else ""))

    retrained_names = res_retrained["doc_preds"].get(doc["filename"], [])
    print(f"  • Retrained Model Extracted ({len(retrained_names)}):")
    print("    " + ", ".join(f"'{p}'" for p in retrained_names[:6]) + ("..." if len(retrained_names) > 6 else ""))